###### Imports and Settings

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import requests
import io
import pickle
import matplotlib.pyplot as plt
from collections import deque
%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)

###### Functions

In [2]:
#function for percent of whole
def percent(x, y):
    try:
        return round((x/y)*100, 2)
    except ZeroDivisionError:
        return 666
#function for population density
def populationdensity(x, y):
    try:
        return round(x/y, 2)
    except ZeroDivisionError:
        return 666

# Comprehensive Plan Data Pull  

The following API calls are designed to streamline the data pulls for the comprehensive plans for any geography. For the sake of simplicity, there are three separate documents for ACS variables, Decennial 2000 SF3 variables, and Decennial SF1 variables. This document contains ACS 5-Year Estimate variables.

In [3]:
#to read in... rb is read bite
with open('api_keys.pkl', 'rb') as keys_file:
        keys_dict_2 = pickle.load(keys_file)

In [4]:
#create a variable that contains your api key
#api_key = keys_dict_2['CENSUS']
api_key = '24fc7d81b74510d599f702dbd408fb18e1466d81'

In [5]:
GNRC = ['161', #Stewart
       '125', #Montgomery
       '083', #Houston
       '085', #Humphreys
       '043', #Dickson
       '021', #Cheatham
       '147', #Robertson
       '165', #Sumner
       '037', #Davidson
       '189', #Wilson
       '169', #Trousdale
       '187', #Williamson
       '149'] #Rutherford

## Read In Data Guide

The "head" should never be more than 2 variables, and the "tail" never more than 2. Since we can pull 50 variables at once this means that we can systematically program in 46 variables for each pull, so that's:
+ dg1: ID's 1 - 46  
+ dg2: ID's 47-92  
+ dg3: ID's 93-138  
+ dg4: ID's 139-184  
+ dg5: ID's 185-230  
+ dg6: ID's 231-276  
+ dg7: ID's 277-322  
+ dg8: ID's 323-368  
+ dg9: ID's 369-414  
+ dg10: ID's 415-460  
+ dg11: ID's 461-506  
+ dg12: ID's 507-552  
+ dg13: ID's 553-598  
+ dg14: ID's 599-644  
+ dg15: ID's 645-690  
+ dg16: ID's 691-736  
+ dg17: ID's 737-782  
+ dg18: ID's 783-828  
**This data guide stops at ID 783 as of 5/18/2022.**

In [6]:
dataguide = pd.read_csv('../DATA GUIDE ACS.csv', dtype = str)
dataguide['ID'] = dataguide['ID'].astype(int)

In [7]:
dg1 = dataguide[dataguide['ID'].between(1, 46)]
dg2 = dataguide[dataguide['ID'].between(47, 92)]
dg3 = dataguide[dataguide['ID'].between(93, 138)]
dg4 = dataguide[dataguide['ID'].between(139, 184)]
dg5 = dataguide[dataguide['ID'].between(185, 230)]
dg6 = dataguide[dataguide['ID'].between(231, 276)]
dg7 = dataguide[dataguide['ID'].between(277, 322)]
dg8 = dataguide[dataguide['ID'].between(323, 368)]
dg9 = dataguide[dataguide['ID'].between(369, 414)]
dg10 = dataguide[dataguide['ID'].between(415, 460)]
dg11 = dataguide[dataguide['ID'].between(461, 506)]
dg12 = dataguide[dataguide['ID'].between(507, 552)]
dg13 = dataguide[dataguide['ID'].between(553, 598)]
dg14 = dataguide[dataguide['ID'].between(599, 644)]
dg15 = dataguide[dataguide['ID'].between(645, 690)]
dg16 = dataguide[dataguide['ID'].between(691, 736)]
dg17 = dataguide[dataguide['ID'].between(737, 782)]
dg18 = dataguide[dataguide['ID'].between(783, 828)]

## Loop through all dataguides and geographies in one API call

In [8]:
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'CountyFIPS'

In [9]:
#dflist = [dg1,dg2,dg3,dg4,dg5,dg6,dg7,dg8,dg9,dg10,dg11,dg12,dg13,dg14,dg15,dg16,dg17,dg18]

In [10]:
dflist = [dg1,dg2, dg3]

In [11]:
var_list = list(dg1['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg1['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)

In [12]:
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str, index = None)
    data['Region'] = 'GNRC'
    data_appended.append(data)
data_appended = pd.concat(data_appended)
data = data_appended

In [13]:
data.head()

,NAME,GEO_ID,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,age_f_70to74,StateFIPS,CountyFIPS,Region
0,"Stewart County, Tennessee",0500000US47161,13553,6572,353,476,364,263,121,50,127,177,438,339,303,456,437,494,459,283,227,209,272,291,191,113,129,6981,422,305,419,278,167,115,126,101,418,401,317,463,420,537,470,232,317,222,247,345,47,161,GNRC
0,"Montgomery County, Tennessee",0500000US47125,204992,102205,8916,7959,7229,4053,2544,1642,1694,5927,10985,9174,7666,6094,5506,5259,4623,2116,2378,1684,1912,1930,1315,866,733,102787,8202,7512,7228,3840,2668,2397,1482,4333,10075,8753,7521,6207,5940,5600,5312,2537,2555,2069,1672,2656,47,125,GNRC
0,"Houston County, Tennessee",0500000US47083,8201,4119,216,347,234,213,204,45,19,53,244,219,186,267,260,256,311,175,106,73,169,237,164,54,67,4082,192,295,190,94,82,51,68,91,262,219,164,289,292,275,295,113,182,89,176,240,47,083,GNRC
0,"Humphreys County, Tennessee",0500000US47085,18528,9113,538,565,663,353,193,74,130,373,595,470,573,502,564,612,685,271,333,198,284,502,398,177,60,9415,424,494,659,308,295,77,69,283,527,486,726,414,595,686,812,179,407,417,275,394,47,085,GNRC
0,"Dickson County, Tennessee",0500000US47043,53289,26246,1733,1662,1850,1196,594,502,204,702,1750,1773,2077,1255,1752,1889,1571,801,1120,615,768,1056,667,460,249,27043,1471,1793,1573,1001,429,387,204,874,1922,1745,2180,1231,1815,1811,1929,774,1206,745,887,1139,47,043,GNRC


In [14]:
api_key = '24fc7d81b74510d599f702dbd408fb18e1466d81'

In [15]:
data = []
for index, element in enumerate(dflist):
    var_list = list(dflist[index]['ACS Variable'])
    #add stuff to variables list
    var_list = deque(var_list)
    var_list.appendleft(head2)
    var_list.appendleft(head1)
    var_list = list(var_list)
    #make columns list
    col_list = list(dflist[index]['Column Name'])
    #add stuff to columns list
    col_list.append(tail_cols1)
    col_list.append(tail_cols2)
    col_list = deque(col_list)
    col_list.appendleft(head2)
    col_list.appendleft(head1)
    col_list = list(col_list)
    #API call
    data_appended = []
    for i in GNRC:
        url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
        predicates= {}
        get_vars= var_list
        predicates["get"]= ",". join(get_vars)
        predicates["for"]= "county:{}".format(i)
        predicates["in"]= "state:47"
        data= requests.get(url_str, params= predicates)
        col_names = col_list
        data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
        data['Region'] = 'GNRC'
        data_appended.append(data)
    data_appended = pd.concat(data_appended)
    dflist[index] = data_appended.reset_index(drop = True)
data = dflist[index-1].merge(dflist[index], how = "outer", on = 'GEO_ID')
print('Your API call is complete.')

ConnectionError: HTTPSConnectionPool(host='api.census.gov', port=443): Max retries exceeded with url: /data/2020/acs/acs5?key=24fc7d81b74510d599f702dbd408fb18e1466d81&get=NAME%2CGEO_ID%2CB01001_047E%2CB01001_048E%2CB01001_049E%2CB01001A_001E%2CB01001B_001E%2CB01001C_001E%2CB01001D_001E%2CB01001E_001E%2CB01001F_001E%2CB01001G_001E%2CB01001H_001E%2CB01001I_001E%2CB25010_001E%2CB27001_001E%2CB27001_002E%2CB27001_003E%2CB27001_004E%2CB27001_005E%2CB27001_006E%2CB27001_007E%2CB27001_008E%2CB27001_009E%2CB27001_010E%2CB27001_011E%2CB27001_012E%2CB27001_013E%2CB27001_014E%2CB27001_015E%2CB27001_016E%2CB27001_017E%2CB27001_018E%2CB27001_019E%2CB27001_020E%2CB27001_021E%2CB27001_022E%2CB27001_023E%2CB27001_024E%2CB27001_025E%2CB27001_026E%2CB27001_027E%2CB27001_028E%2CB27001_029E%2CB27001_030E%2CB27001_031E%2CB27001_032E%2CB27001_033E&for=county%3A037&in=state%3A47 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x00000221740A3130>: Failed to establish a new connect

In [ ]:
    var_list.appendleft(head1)
    var_list = list(var_list)
    #make columns list
    col_list = list(dflist[index]['Column Name'])
    #add stuff to columns list
    col_list.append(tail_cols1)
    col_list.append(tail_cols2)
    col_list = deque(col_list)
    col_list.appendleft(head2)
    col_list.appendleft(head1)
    col_list = list(col_list)
    #API call
    data_appended = []
    for i in GNRC:
        url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
        predicates= {}
        get_vars= var_list
        predicates["get"]= ",". join(get_vars)
        predicates["for"]= "county:{}".format(i)
        predicates["in"]= "state:47"
        data= requests.get(url_str, params= predicates, timeout=10)
        col_names = col_list
        data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
        data['Region'] = 'GNRC'
        data_appended.append(data)
    data_appended = pd.concat(data_appended)
    dflist[index] = data_appended.reset_index(drop = True)
data = dflist[index-1].merge(dflist[index], how = "outer", on = 'GEO_ID')
print('Your API call is complete.')

In [ ]:
data.head()

In [ ]:
data_appended.head()

In [ ]:
data.head()

In [ ]:
# THIS WORKS FOR TWO DATA GUIDES IN THE LIST~!!!!!!!!!!!!!!
for index, element in enumerate(dflist):
    var_list = list(dflist[index]['ACS Variable'])
    #add stuff to variables list
    var_list = deque(var_list)
    var_list.appendleft(head2)
    var_list.appendleft(head1)
    var_list = list(var_list)
    #make columns list
    col_list = list(dflist[index]['Column Name'])
    #add stuff to columns list
    col_list.append(tail_cols1)
    col_list.append(tail_cols2)
    col_list = deque(col_list)
    col_list.appendleft(head2)
    col_list.appendleft(head1)
    col_list = list(col_list)
    #API call
    data_appended = []
    for i in GNRC:
        url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
        predicates= {}
        get_vars= var_list
        predicates["get"]= ",". join(get_vars)
        predicates["for"]= "county:{}".format(i)
        predicates["in"]= "state:47"
        data= requests.get(url_str, params= predicates)
        col_names = col_list
        data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
        data['Region'] = 'GNRC'
        data_appended.append(data)
    data_appended = pd.concat(data_appended)
    dflist[index] = data_appended.reset_index(drop = True)
data = dflist[index-1].merge(dflist[index], how = "outer", on = 'GEO_ID')
print('Your API call is complete.')

In [41]:
# THIS WORKS FOR LAST TWO ELEMENTS IN LIST
for index, element in enumerate(dflist):
    var_list = list(dflist[index]['ACS Variable'])
    #add stuff to variables list
    var_list = deque(var_list)
    var_list.appendleft(head2)
    var_list.appendleft(head1)
    var_list = list(var_list)
    #make columns list
    col_list = list(dflist[index]['Column Name'])
    #add stuff to columns list
    col_list.append(tail_cols1)
    col_list.append(tail_cols2)
    col_list = deque(col_list)
    col_list.appendleft(head2)
    col_list.appendleft(head1)
    col_list = list(col_list)
    #API call
    data_appended = []
    for i in GNRC:
        url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
        predicates= {}
        get_vars= var_list
        predicates["get"]= ",". join(get_vars)
        predicates["for"]= "county:{}".format(i)
        predicates["in"]= "state:47"
        data= requests.get(url_str, params= predicates)
        col_names = col_list
        data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
        data['Region'] = 'GNRC'
        data_appended.append(data)
    data_appended = pd.concat(data_appended)
    dflist[index] = data_appended.reset_index(drop = True)
data = dflist[index-1].merge(dflist[index], how = "outer", on = 'GEO_ID')
data_appended = data
print('Your API call is complete.')

ConnectionError: HTTPSConnectionPool(host='api.census.gov', port=443): Max retries exceeded with url: /data/2020/acs/acs5?key=24fc7d81b74510d599f702dbd408fb18e1466d81&get=NAME%2CGEO_ID%2CB01001_001E%2CB01001_002E%2CB01001_003E%2CB01001_004E%2CB01001_005E%2CB01001_006E%2CB01001_007E%2CB01001_008E%2CB01001_009E%2CB01001_010E%2CB01001_011E%2CB01001_012E%2CB01001_013E%2CB01001_014E%2CB01001_015E%2CB01001_016E%2CB01001_017E%2CB01001_018E%2CB01001_019E%2CB01001_020E%2CB01001_021E%2CB01001_022E%2CB01001_023E%2CB01001_024E%2CB01001_025E%2CB01001_026E%2CB01001_027E%2CB01001_028E%2CB01001_029E%2CB01001_030E%2CB01001_031E%2CB01001_032E%2CB01001_033E%2CB01001_034E%2CB01001_035E%2CB01001_036E%2CB01001_037E%2CB01001_038E%2CB01001_039E%2CB01001_040E%2CB01001_041E%2CB01001_042E%2CB01001_043E%2CB01001_044E%2CB01001_045E%2CB01001_046E&for=county%3A169&in=state%3A47 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x00000189DEAF9340>: Failed to establish a new connection: [Win

In [ ]:
data.head()

In [ ]:
data_appended.head()

In [ ]:
c3 = c3.drop(columns = [head1, tail_cols1, tail_cols2])
c4 = c4.drop(columns = [head1, tail_cols1, tail_cols2])
c5 = c5.drop(columns = [head1, tail_cols1, tail_cols2])
c6 = c6.drop(columns = [head1, tail_cols1, tail_cols2])
c7 = c7.drop(columns = [head1, tail_cols1, tail_cols2])
c8 = c8.drop(columns = [head1, tail_cols1, tail_cols2])
c9 = c9.drop(columns = [head1, tail_cols1, tail_cols2])
c10 = c10.drop(columns = [head1, tail_cols1, tail_cols2])
c11 = c11.drop(columns = [head1, tail_cols1, tail_cols2])
c12 = c12.drop(columns = [head1, tail_cols1, tail_cols2])
c13 = c13.drop(columns = [head1, tail_cols1, tail_cols2])
c14 = c14.drop(columns = [head1, tail_cols1, tail_cols2])
c15 = c15.drop(columns = [head1, tail_cols1, tail_cols2])
c16 = c16.drop(columns = [head1, tail_cols1, tail_cols2])
c17 = c17.drop(columns = [head1, tail_cols1, tail_cols2])
c18 = c18.drop(columns = [head1, tail_cols1, tail_cols2])

In [ ]:
dfs = [c2, c3, c4, c5, c6, c7, c8, c9, c10, c11, c12, c13, c14, c15, c16, c17, finalcall]

In [ ]:
df_merged = reduce(lambda  left,right: pd.merge(left,right,on=['GEO_ID'],
                                            how='outer'), dfs)